In [ ]:
import numpy as np
import defs, propagation
from NuRadioMC.utilities.medium import greenland_simple

from warnings import filterwarnings
filterwarnings("ignore", "invalid value encountered in subtract")

# 1. Set up computational geometry.

## 1.1. Build grid.

In [ ]:
rmax = 1000 # rmin always assumed 0
zmin, zmax = -500, 500
dr, dz = 1, 1   # grid spacing
npts_r, npts_z = int(rmax / dr) + 1, int((zmax - zmin) / dz) + 1
tx_pos = np.array([0, -60]) # Source coordinate

## 1.2. Define refractive index profile.

See `tt_example.py` for an example using built-in refractive index functions. In this example, we show how to use the `greenland_simple` ice model from NuRadioMC. Applicable to any non-birefringent ice model.

In [ ]:
ice = greenland_simple()
ior, grad_ior = defs.get_ior_from_nuradio(ice)
reflection_at_z = ice.z_air_boundary

# 2. Run solver and save results.

## 2.1. Initialize solver.

In [ ]:
ttc = propagation.TravelTimeCalculator(tx_pos[1], zmin, zmax, rmax, npts_z, npts_r)
nrays = 0  # Set number of big rays for refracted maps
ttc.set_ior_and_solve(ior, grad_ior, nrays, reflection_at_z)
print(ttc.travel_time_fields)

Note that traveltimes are written as pykonal ScalarField3D objects. Most methods have wrappers written in reconal (shown below); see pykonal documentation [here](https://malcolmw.github.io/pykonal-docs/api/classes/fields/ScalarField3D.html) for all available methods.

## 2.2. Save results to disk.

Maps are saved as numpy .npz files, with one array for each component of the multivalued traveltime field and one array for metadata. The option is provided to compress the file if desired.

In [ ]:
ttc.save_to_disk('tt maps.npz', compressed = False)

# 3. Use results.

## 3.1. Load results from disk.

In [ ]:
new_ttc = propagation.TravelTimeCalculator.FromFile('tt maps.npz')

## 3.2. Extract traveltimes and tangent vectors.

Find direct traveltime at a set of arbitrary coordinates within domain. Zero- and first-order approximations are available; they agree at exact nodes.

In [ ]:
comp = 'direct'
coords = np.array([[247., -192.],
                  [542.1, 157.9],
                  [3.2, -493.1],
                  [982.67, 416.7]
                ])

# Zero-order approximation.
tt_zero = new_ttc.get_travel_time(coords, comp = 'direct', order = 'zero')
print(tt_zero)

# First-order approximation.
tt_first = new_ttc.get_travel_time(coords, comp = 'direct', order = 'first')
print(tt_first)

Use traveltime gradients to find the direct launch vector at a given coordinate.

In [ ]:
# Zero-order approximation.
vecs_zero = new_ttc.get_tangent_vector(coords, comp = 'direct', order = 'zero', unit = True)
print(vecs_zero)

# First-order approximation.
vecs_first = new_ttc.get_tangent_vector(coords, comp = 'direct', order = 'first', unit = True)
print(vecs_first)